<a href="https://colab.research.google.com/github/J-Cos/EuropeanRewildingProgress/blob/main/MODIS_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MODIS NDVI Annual Metrics: INDVI, minNDVI, maxNDVI
## Overview
This notebook computes three annual NDVI metrics from MODIS satellite data for a 10×10 km example site.
- **INDVI** (Integrated NDVI) — a proxy for annual vegetation productivity, calculated as mean NDVI × number of compositing periods per year.
- **minNDVI** — the minimum NDVI value recorded in each year, reflecting baseline greenness.
- **maxNDVI** — the peak NDVI value recorded in each year, reflecting maximum canopy cover.
## Data Sources
| Product | Resolution | Temporal | ID |
|---------|-----------|----------|----|
| MOD13Q1 (primary) | 250 m | 16-day | `MODIS/061/MOD13Q1` |
| MOD13A3 (sensitivity) | 1000 m | Monthly | `MODIS/061/MOD13A3` |
## Preprocessing (250 m only)
1. **Quality filtering**: Pixels flagged as cloud-contaminated, snow/ice, or shadow are masked using both the `SummaryQA` band (retaining only "Good" and "Marginal" data) and individual `DetailedQA` bit flags (bits 10, 14, 15).
2. **Temporal smoothing**: Following the method of Garonna et al. (2009), as applied by Schulte to Bühne et al (2022) and Williams et al. (2024):
   - Single outliers (≥0.3 NDVI units above or below both neighbouring composites) are replaced with the mean of those neighbours.
   - Two consecutive contaminated values are replaced with the mean of the closest clean values either side of the pair.
## Sensitivity Analysis (1000 m)
The 1000 m monthly product is processed **without** QA filtering or smoothing, following Williams et al. (2024), to assess whether results are sensitive to the preprocessing workflow.
## Output
Each resolution produces an `ee.ImageCollection` of three images (`INDVI`, `minNDVI`, `maxNDVI`), where each image contains one band per year (e.g. `INDVI_2000`, `INDVI_2001`, …).
## NOTE
Agentic programming was used to refactor this analysis to Python API, full unit testing is applied.

## 1. Setup & Study Area
(change to your project ID)

In [ ]:
import ee
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from datetime import datetime
import geemap


# Authenticate and initialize
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='quantum-bonus-434714-t2')
# Dummy 10x10km site roughly over the top of Knepp Estate
roi = ee.Geometry.Point([-0.36, 50.96]).buffer(5000).bounds()
# Set the year range (adjust to your available years)
start_year = 2000
end_year = 2025

## 2. Function Definitions

In [ ]:
def prep_modis(image):
    """QA-mask clouds, snow, shadow and scale NDVI to float."""
    qa = image.select('SummaryQA')
    qa_mask = qa.lte(1)
    dqa = image.select('DetailedQA')
    mask = qa_mask \
        .And(dqa.bitwiseAnd(1 << 10).eq(0)) \
        .And(dqa.bitwiseAnd(1 << 14).eq(0)) \
        .And(dqa.bitwiseAnd(1 << 15).eq(0))
    ndvi = image.select('NDVI').multiply(0.0001).rename('NDVI')
    return ndvi.updateMask(mask) \
        .set('system:time_start', image.get('system:time_start'))


def garonna_smooth(collection):
    """
    Garonna et al. 2009 smoothing with change reporting.
    Returns a dict: {'collection': smoothed IC, 'changes': ee.Dictionary}
    """
    img_list = collection.toList(collection.size())
    n = collection.size()
    indices = ee.List.sequence(0, n.subtract(1))

    def _ndvi(i):
        return ee.Image(img_list.get(
            ee.Number(i).max(0).min(n.subtract(1))
        )).select('NDVI')

    def _flag(boolean_num):
        return ee.Image.constant(
            ee.Number(ee.Algorithms.If(boolean_num, 1, 0)))

    def _smooth(index):
        index = ee.Number(index)
        cur = ee.Image(img_list.get(index))
        v  = cur.select('NDVI')
        p1 = _ndvi(index.subtract(1))
        n1 = _ndvi(index.add(1))
        p2 = _ndvi(index.subtract(2))
        n2 = _ndvi(index.add(2))

        dp = v.subtract(p1)
        dn = v.subtract(n1)
        is_single = (dp.gte(0.3).And(dn.gte(0.3))) \
                 .Or(dp.lte(-0.3).And(dn.lte(-0.3)))
        is_single = is_single.multiply(
            _flag(index.gt(0).And(index.lt(n.subtract(1)))))

        is_pair1 = v.subtract(p1).abs().gte(0.3) \
            .And(n1.subtract(v).abs().lt(0.3)) \
            .And(n2.subtract(n1).abs().gte(0.3))
        is_pair1 = is_pair1.multiply(
            _flag(index.gt(0).And(index.lt(n.subtract(2)))))

        is_pair2 = p1.subtract(p2).abs().gte(0.3) \
            .And(v.subtract(p1).abs().lt(0.3)) \
            .And(n1.subtract(v).abs().gte(0.3))
        is_pair2 = is_pair2.multiply(
            _flag(index.gt(1).And(index.lt(n.subtract(1)))))

        any_change = is_single.Or(is_pair1).Or(is_pair2).rename('changed')

        smoothed = v
        smoothed = smoothed.where(is_single, p1.add(n1).divide(2))
        smoothed = smoothed.where(is_pair1,  p1.add(n2).divide(2))
        smoothed = smoothed.where(is_pair2,  p2.add(n1).divide(2))

        out = cur.addBands(smoothed.rename('NDVI'), overwrite=True)
        # Carry the change flag as an extra band
        return out.addBands(any_change)

    smoothed_col = ee.ImageCollection(indices.map(_smooth))

    # Count total changed pixels across all timesteps within the ROI
    total_changed = smoothed_col.select('changed') \
        .sum().reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=roi,
            scale=250,
            maxPixels=1e9
        )

    # Return the clean collection (without the flag band) + the counts
    clean_col = smoothed_col.select('NDVI')
    return {
        'collection': clean_col,
        'total_changed_pixels': total_changed
    }

def compute_annual_metrics(collection, start, end, periods_per_year):
    """INDVI (integral proxy = mean * periods), minNDVI, maxNDVI per year."""
    years = ee.List.sequence(start, end)
    def _year(y):
        y = ee.Number(y)
        yc = collection.filter(ee.Filter.calendarRange(y, y, 'year'))
        return ee.Image.cat(
            yc.mean().multiply(periods_per_year).rename('INDVI'),
            yc.min().rename('minNDVI'),
            yc.max().rename('maxNDVI')
        ).set('year', y,
              'system:time_start', ee.Date.fromYMD(y, 1, 1).millis())
    return ee.ImageCollection(years.map(_year))

def scale_only(image):
    """Scale NDVI to -1..1 range, no QA masking."""
    ndvi = image.select('NDVI').multiply(0.0001).rename('NDVI')
    return ndvi.set('system:time_start', image.get('system:time_start'))

def stack_metric(annual_col, metric):
    """One image per metric, with year-labelled bands."""
    selected = annual_col.select(metric)
    stacked = selected.toBands()
    names = annual_col.aggregate_array('year').map(
        lambda y: ee.String(metric).cat('_').cat(
            ee.Number(y).format('%04d')))
    return stacked.rename(names)

def build_final_collection(annual_col):
    """Stack each metric into a single multi-year-banded image."""
    return ee.ImageCollection([
        stack_metric(annual_col, m).set('metric', m) for m in METRICS
    ])

## 3. Unit Tests
(Essential for agentic programming)

In [ ]:
def run_all_tests():
    pt = ee.Geometry.Point([0, 0])

    def make_synth(vals):
        def _make(i):
            i = ee.Number(i)
            return ee.Image.constant(ee.Number(ee.List(vals).get(i))) \
                .toFloat().rename('NDVI') \
                .set('system:time_start',
                     ee.Date('2020-01-01').advance(i.multiply(16), 'day').millis())
        return ee.ImageCollection(
            ee.List.sequence(0, len(vals) - 1).map(_make))

    def sample_at(collection, index, point=pt):
        return ee.Image(collection.toList(collection.size()).get(index)) \
            .sample(point, 1000).first().get('NDVI').getInfo()

    # TEST 1a: single spike down
    sm1 = garonna_smooth(make_synth([0.5, 0.1, 0.5, 0.5, 0.5]))['collection']
    t1 = sample_at(sm1, 1)
    assert abs(t1 - 0.5) < 0.01, f'FAIL: {t1}'
    assert abs(sample_at(sm1, 0) - 0.5) < 0.01
    assert abs(sample_at(sm1, 2) - 0.5) < 0.01
    print(f'PASS 1a — single spike down: t1 corrected from 0.1 → {t1:.4f}')

    # TEST 1b: single spike up
    sm1b = garonna_smooth(make_synth([0.3, 0.8, 0.3, 0.3, 0.3]))['collection']
    t1b = sample_at(sm1b, 1)
    assert abs(t1b - 0.3) < 0.01, f'FAIL: {t1b}'
    print(f'PASS 1b — single spike up:   t1 corrected from 0.8 → {t1b:.4f}')

    # TEST 2: consecutive pair
    sm2 = garonna_smooth(make_synth([0.5, 0.1, 0.15, 0.5, 0.5]))['collection']
    t1p = sample_at(sm2, 1)
    t2p = sample_at(sm2, 2)
    assert abs(t1p - 0.5) < 0.01, f'FAIL t1: {t1p}'
    assert abs(t2p - 0.5) < 0.01, f'FAIL t2: {t2p}'
    print(f'PASS 2  — consecutive pair:  t1={t1p:.4f}, t2={t2p:.4f}')

    # TEST 3: clean data unchanged
    clean = [0.4, 0.5, 0.45, 0.48, 0.5]
    sm3 = garonna_smooth(make_synth(clean))['collection']
    for i, exp in enumerate(clean):
        v = sample_at(sm3, i)
        assert abs(v - exp) < 0.01, f'FAIL t{i}: expected {exp}, got {v}'
    print(f'PASS 3  — clean data unchanged: all 5 values preserved')

    # TEST 4: change count > 0
    res4 = garonna_smooth(make_synth([0.5, 0.1, 0.5, 0.5, 0.5]))
    count = res4['total_changed_pixels'].getInfo()['changed']
    assert count > 0, f'FAIL: {count}'
    print(f'PASS 4  — change count reported: {count:.1f}')

    # TEST 5: boundary images untouched
    sm5 = garonna_smooth(make_synth([0.1, 0.5, 0.5, 0.5, 0.9]))['collection']
    assert abs(sample_at(sm5, 0) - 0.1) < 0.01
    assert abs(sample_at(sm5, 4) - 0.9) < 0.01
    print(f'PASS 5  — boundary values untouched')

    # TEST 6: sub-threshold NOT smoothed
    sm6 = garonna_smooth(make_synth([0.5, 0.21, 0.5, 0.5, 0.5]))['collection']
    t1s = sample_at(sm6, 1)
    assert abs(t1s - 0.21) < 0.01, f'FAIL: {t1s}'
    print(f'PASS 6  — sub-threshold (0.29 diff) not smoothed: t1={t1s:.4f}')

    # TEST 7: compute_annual_metrics
    test_vals = [0.2, 0.4, 0.6, 0.8, 0.6, 0.4]
    def make_annual(i):
        i = ee.Number(i)
        return ee.Image.constant(ee.Number(ee.List(test_vals).get(i))) \
            .toFloat().rename('NDVI') \
            .set('system:time_start',
                 ee.Date('2020-01-01').advance(i.multiply(60), 'day').millis())
    annual_synth = ee.ImageCollection(
        ee.List.sequence(0, len(test_vals) - 1).map(make_annual))
    metrics = compute_annual_metrics(annual_synth, 2020, 2020, len(test_vals))
    m = ee.Image(metrics.first()).sample(pt, 1000).first().getInfo()['properties']
    exp_indvi = sum(test_vals) / len(test_vals) * len(test_vals)  # 3.0
    assert abs(m['INDVI'] - exp_indvi) < 0.01, f'FAIL INDVI: {m["INDVI"]}'
    assert abs(m['minNDVI'] - 0.2) < 0.01, f'FAIL min: {m["minNDVI"]}'
    assert abs(m['maxNDVI'] - 0.8) < 0.01, f'FAIL max: {m["maxNDVI"]}'
    print(f'PASS 7  — annual metrics: INDVI={m["INDVI"]:.2f}, min={m["minNDVI"]:.2f}, max={m["maxNDVI"]:.2f}')

    # TEST 8: stack_metric band names and values
    def make_year_img(y):
        y = ee.Number(y)
        return ee.Image.constant(y.subtract(2019).multiply(0.1)).toFloat() \
            .rename('INDVI') \
            .addBands(ee.Image.constant(0.1).toFloat().rename('minNDVI')) \
            .addBands(ee.Image.constant(0.9).toFloat().rename('maxNDVI')) \
            .set('year', y,
                 'system:time_start', ee.Date.fromYMD(y, 1, 1).millis())
    dummy_annual = ee.ImageCollection(
        ee.List([2020, 2021, 2022]).map(make_year_img))
    stacked = stack_metric(dummy_annual, 'INDVI')
    bands = stacked.bandNames().getInfo()
    assert bands == ['INDVI_2020', 'INDVI_2021', 'INDVI_2022'], f'FAIL: {bands}'
    sv = stacked.sample(pt, 1000).first().getInfo()['properties']
    assert abs(sv['INDVI_2020'] - 0.1) < 0.01
    assert abs(sv['INDVI_2021'] - 0.2) < 0.01
    assert abs(sv['INDVI_2022'] - 0.3) < 0.01
    print(f'PASS 8  — stack_metric: bands={bands}, values correct')

    # TEST 9: prep_modis scales and masks
    test_img = ee.ImageCollection('MODIS/061/MOD13Q1').filterBounds(roi).first()
    prepped = prep_modis(test_img)
    assert prepped.bandNames().getInfo() == ['NDVI']
    stats = prepped.reduceRegion(
        reducer=ee.Reducer.minMax(), geometry=roi,
        scale=250, maxPixels=1e9).getInfo()
    assert -1.0 <= stats['NDVI_min'] <= 1.0
    assert -1.0 <= stats['NDVI_max'] <= 1.0
    print(f'PASS 9  — prep_modis: NDVI scaled to [{stats["NDVI_min"]:.3f}, {stats["NDVI_max"]:.3f}]')

    # TEST 10: scale_only — scales but does NOT mask
    test_img_raw = ee.ImageCollection('MODIS/061/MOD13A3').filterBounds(roi).first()
    scaled = scale_only(test_img_raw)
    assert scaled.bandNames().getInfo() == ['NDVI'], \
        f'FAIL scale_only bands: {scaled.bandNames().getInfo()}'
    # Confirm no masking was applied — pixel count should equal total pixel count
    unmasked = scaled.reduceRegion(
        reducer=ee.Reducer.count(), geometry=roi,
        scale=1000, maxPixels=1e9).getInfo()
    total = test_img_raw.select('NDVI').reduceRegion(
        reducer=ee.Reducer.count(), geometry=roi,
        scale=1000, maxPixels=1e9).getInfo()
    assert unmasked['NDVI'] == total['NDVI'], \
        f'FAIL scale_only masking: {unmasked["NDVI"]} != {total["NDVI"]}'
    # Confirm scaling: raw value * 0.0001 = scaled value
    raw_val = test_img_raw.select('NDVI').reduceRegion(
        reducer=ee.Reducer.first(), geometry=roi,
        scale=1000, maxPixels=1e9).getInfo()['NDVI']
    scaled_val = scaled.reduceRegion(
        reducer=ee.Reducer.first(), geometry=roi,
        scale=1000, maxPixels=1e9).getInfo()['NDVI']
    assert abs(scaled_val - raw_val * 0.0001) < 0.0001, \
        f'FAIL scale_only value: {scaled_val} != {raw_val} * 0.0001'
    print(f'PASS 10 — scale_only: no masking, scaled {raw_val} → {scaled_val:.4f}')

    # TEST 11: build_final_collection — produces 3 images with correct metrics
    def make_year_img_11(y):
        y = ee.Number(y)
        return ee.Image.constant(0.5).toFloat().rename('INDVI') \
            .addBands(ee.Image.constant(0.2).toFloat().rename('minNDVI')) \
            .addBands(ee.Image.constant(0.8).toFloat().rename('maxNDVI')) \
            .set('year', y,
                 'system:time_start', ee.Date.fromYMD(y, 1, 1).millis())
    dummy = ee.ImageCollection(ee.List([2020, 2021]).map(make_year_img_11))
    final = build_final_collection(dummy)

    assert final.size().getInfo() == 3, \
        f'FAIL build_final: expected 3 images, got {final.size().getInfo()}'
    metric_labels = final.aggregate_array('metric').getInfo()
    assert set(metric_labels) == {'INDVI', 'minNDVI', 'maxNDVI'}, \
        f'FAIL build_final metrics: {metric_labels}'
    # Check band count on first image (should be 2 years)
    first_bands = final.first().bandNames().getInfo()
    assert len(first_bands) == 2, \
        f'FAIL build_final band count: expected 2, got {len(first_bands)}'
    print(f'PASS 11 — build_final_collection: 3 images, metrics={metric_labels}, '
          f'bands per image={len(first_bands)}')

    print('\n✅ All 11 tests passed!')

run_all_tests()


PASS 1a — single spike down: t1 corrected from 0.1 → 0.5000
PASS 1b — single spike up:   t1 corrected from 0.8 → 0.3000
PASS 2  — consecutive pair:  t1=0.5000, t2=0.5000
PASS 3  — clean data unchanged: all 5 values preserved
PASS 4  — change count reported: 2527.6
PASS 5  — boundary values untouched
PASS 6  — sub-threshold (0.29 diff) not smoothed: t1=0.2100
PASS 7  — annual metrics: INDVI=3.00, min=0.20, max=0.80
PASS 8  — stack_metric: bands=['INDVI_2020', 'INDVI_2021', 'INDVI_2022'], values correct
PASS 9  — prep_modis: NDVI scaled to [0.196, 0.828]
PASS 10 — scale_only: no masking, scaled 7065 → 0.7065
PASS 11 — build_final_collection: 3 images, metrics=['INDVI', 'minNDVI', 'maxNDVI'], bands per image=2

✅ All 11 tests passed!


##4. Load Data

In [ ]:
col_250 = ee.ImageCollection('MODIS/061/MOD13Q1') \
    .filterBounds(roi) \
    .filterDate(str(start_year) + '-01-01', str(end_year) + '-12-31')

col_1000 = ee.ImageCollection('MODIS/061/MOD13A3') \
    .filterBounds(roi) \
    .filterDate(str(start_year) + '-01-01', str(end_year) + '-12-31')

##5. Preprocess 250m and calculate metrics (MOD13Q1)





In [ ]:
result_250 = garonna_smooth(col_250.map(prep_modis))
smooth_250 = result_250['collection']
print('250m changed pixels:', result_250['total_changed_pixels'].getInfo())
annual_250 = compute_annual_metrics(smooth_250, start_year, end_year, 23)
print('250m years:', annual_250.size().getInfo())


250m changed pixels: {'changed': 558.0235294117649}
250m years: 26


## 6. Diagnostic: Inspect Smoothing Corrections by Year
(Functions not unit tested)


In [ ]:
# Diagnostic: show per-year change counts
smooth_col_with_flags = garonna_smooth(col_250.map(prep_modis))
flagged = smooth_col_with_flags['collection']

# Re-run but keep the change flags this time
img_list = col_250.map(prep_modis).toList(col_250.size())
n = col_250.map(prep_modis).size()

# Get the flagged collection before we stripped the 'changed' band
indices = ee.List.sequence(0, n.subtract(1))

def _ndvi(i):
    return ee.Image(img_list.get(
        ee.Number(i).max(0).min(n.subtract(1))
    )).select('NDVI')

def _flag(boolean_num):
    return ee.Image.constant(
        ee.Number(ee.Algorithms.If(boolean_num, 1, 0)))

def _flag_only(index):
    index = ee.Number(index)
    cur = ee.Image(img_list.get(index))
    v  = cur.select('NDVI')
    p1 = _ndvi(index.subtract(1))
    n1 = _ndvi(index.add(1))
    p2 = _ndvi(index.subtract(2))
    n2 = _ndvi(index.add(2))
    dp = v.subtract(p1)
    dn = v.subtract(n1)
    is_single = (dp.gte(0.3).And(dn.gte(0.3))).Or(dp.lte(-0.3).And(dn.lte(-0.3)))
    is_single = is_single.multiply(_flag(index.gt(0).And(index.lt(n.subtract(1)))))
    is_pair1 = v.subtract(p1).abs().gte(0.3).And(n1.subtract(v).abs().lt(0.3)).And(n2.subtract(n1).abs().gte(0.3))
    is_pair1 = is_pair1.multiply(_flag(index.gt(0).And(index.lt(n.subtract(2)))))
    is_pair2 = p1.subtract(p2).abs().gte(0.3).And(v.subtract(p1).abs().lt(0.3)).And(n1.subtract(v).abs().gte(0.3))
    is_pair2 = is_pair2.multiply(_flag(index.gt(1).And(index.lt(n.subtract(1)))))
    changed = is_single.Or(is_pair1).Or(is_pair2).rename('changed')
    return changed.set('system:time_start', cur.get('system:time_start'))

change_flags = ee.ImageCollection(indices.map(_flag_only))

# Sum changes per year
years = ee.List.sequence(start_year, end_year)
def changes_per_year(y):
    y = ee.Number(y)
    yr_flags = change_flags.filter(ee.Filter.calendarRange(y, y, 'year'))
    total = yr_flags.sum().reduceRegion(
        reducer=ee.Reducer.sum(), geometry=roi, scale=250, maxPixels=1e9)
    return ee.Feature(None, {'year': y, 'changes': total.get('changed')})

yearly = ee.FeatureCollection(years.map(changes_per_year))
import pandas as pd
df = pd.DataFrame(yearly.getInfo()['features'])
df = pd.DataFrame([f['properties'] for f in yearly.getInfo()['features']])
print(df.to_string(index=False))


  changes  year
42.796078  2000
 3.623529  2001
27.466667  2002
33.294118  2003
26.349020  2004
13.247059  2005
 4.000000  2006
 5.349020  2007
15.741176  2008
78.164706  2009
15.494118  2010
10.623529  2011
18.219608  2012
10.811765  2013
47.000000  2014
14.596078  2015
22.494118  2016
19.098039  2017
11.047059  2018
20.000000  2019
 4.741176  2020
29.847059  2021
58.396078  2022
11.000000  2023
 4.000000  2024
10.623529  2025


## 7. 1000 m scaling and calculate metrics for Sensitivity Analysis (MOD13A3)

In [ ]:
# 1000m sensitivity: NO preprocessing, NO smoothing (Williams et al. approach)
# The monthly composite is used raw — just scale NDVI to float
raw_1000 = col_1000.map(scale_only)
annual_1000 = compute_annual_metrics(raw_1000, start_year, end_year, 12)
print('1000m years:', annual_1000.size().getInfo())


1000m years: 26


## 8. Build Stacked Output Images (INDVI, minNDVI, maxNDVI)

In [ ]:

METRICS = ['INDVI', 'minNDVI', 'maxNDVI']
final_250  = build_final_collection(annual_250)
final_1000 = build_final_collection(annual_1000)
print('250m INDVI bands:', final_250.first().bandNames().getInfo())
print('1000m INDVI bands:', final_1000.first().bandNames().getInfo())


250m INDVI bands: ['INDVI_2000', 'INDVI_2001', 'INDVI_2002', 'INDVI_2003', 'INDVI_2004', 'INDVI_2005', 'INDVI_2006', 'INDVI_2007', 'INDVI_2008', 'INDVI_2009', 'INDVI_2010', 'INDVI_2011', 'INDVI_2012', 'INDVI_2013', 'INDVI_2014', 'INDVI_2015', 'INDVI_2016', 'INDVI_2017', 'INDVI_2018', 'INDVI_2019', 'INDVI_2020', 'INDVI_2021', 'INDVI_2022', 'INDVI_2023', 'INDVI_2024', 'INDVI_2025']
1000m INDVI bands: ['INDVI_2000', 'INDVI_2001', 'INDVI_2002', 'INDVI_2003', 'INDVI_2004', 'INDVI_2005', 'INDVI_2006', 'INDVI_2007', 'INDVI_2008', 'INDVI_2009', 'INDVI_2010', 'INDVI_2011', 'INDVI_2012', 'INDVI_2013', 'INDVI_2014', 'INDVI_2015', 'INDVI_2016', 'INDVI_2017', 'INDVI_2018', 'INDVI_2019', 'INDVI_2020', 'INDVI_2021', 'INDVI_2022', 'INDVI_2023', 'INDVI_2024', 'INDVI_2025']


## 9. Visualise Annual Metrics (geemap)

In [ ]:
# --- Map 1: Latest year side-by-side 250m vs 1000m ---
Map = geemap.Map(center=[50.96, -0.36], zoom=13)

latest_year = end_year
ndvi_vis = {'min': 0, 'max': 1, 'palette': ['#d73027', '#fee08b', '#1a9850']}
indvi_vis = {'min': 0, 'max': 23, 'palette': ['#ffffcc', '#41b6c4', '#253494']}

# 250 m layers for latest year
latest_250 = annual_250.filter(ee.Filter.eq('year', latest_year)).first()
Map.addLayer(latest_250.select('maxNDVI').clip(roi), ndvi_vis, f'maxNDVI 250m ({latest_year})')
Map.addLayer(latest_250.select('minNDVI').clip(roi), ndvi_vis, f'minNDVI 250m ({latest_year})')
Map.addLayer(latest_250.select('INDVI').clip(roi), indvi_vis, f'INDVI 250m ({latest_year})')

# 1000 m layers for latest year
latest_1000 = annual_1000.filter(ee.Filter.eq('year', latest_year)).first()
Map.addLayer(latest_1000.select('maxNDVI').clip(roi), ndvi_vis, f'maxNDVI 1000m ({latest_year})', shown=False)
Map.addLayer(latest_1000.select('minNDVI').clip(roi), ndvi_vis, f'minNDVI 1000m ({latest_year})', shown=False)
Map.addLayer(latest_1000.select('INDVI').clip(roi), indvi_vis, f'INDVI 1000m ({latest_year})', shown=False)

# ROI outline
Map.addLayer(roi, {'color': 'white'}, 'Study Area')
Map.add_colorbar(ndvi_vis, label='NDVI', layer_name=f'maxNDVI 250m ({latest_year})')

Map


Map(center=[50.96, -0.36], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGU…